# Chapter 5: Modeling Distributions
**Spine:** Think Stats by Allen Downey (adapted for applied analytics work)  
**Libraries:** Production-grade only — `numpy`, `pandas`, `scipy.stats`, `matplotlib`, `seaborn`

---
## What does it mean to model a distribution?

In Chapters 3 and 4 we described data — PMFs and CDFs told us what our sample looks like.

This chapter asks a different question: **what theoretical distribution does our data come from?**

If we can identify the theoretical distribution, we can:
- Make probability statements about values we have not observed
- Build more accurate predictive models
- Choose the right statistical tests downstream
- Communicate uncertainty rigorously to stakeholders

The workflow is always:
1. Look at the data visually — histogram, CDF
2. Hypothesize a theoretical distribution based on what you see and domain knowledge
3. Fit the distribution to the data — estimate its parameters
4. Compare the fitted theoretical CDF to the empirical CDF
5. Test the fit formally with a goodness of fit test

**Three distributions covered in this notebook:**
- **Log-normal** — income, house prices, revenue, session duration
- **Exponential** — time between events, inter-arrival times
- **Normal** — measurement errors, aggregated metrics, Central Limit Theorem

In [ ]:
import ssl
import certifi
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.datasets import fetch_california_housing

# Fix SSL certificate verification on Mac
ssl._create_default_https_context = ssl.create_default_context(cafile=certifi.where())

# Consistent style for all plots
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (9, 4)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

# Load California Housing — used throughout this notebook
housing = fetch_california_housing(as_frame=True).frame

print('All libraries loaded successfully.')
print(f'California Housing dataset: {housing.shape[0]:,} rows, {housing.shape[1]} columns')

---
## Utility functions
Reused throughout this notebook.

In [ ]:
def plot_fit(data: np.ndarray, dist, params: tuple, title: str, xlabel: str):
    """
    Plot empirical CDF vs theoretical CDF side by side with a histogram.
    Production-safe: uses only scipy and matplotlib.
    
    Parameters
    ----------
    data   : raw observations
    dist   : scipy.stats continuous distribution object
    params : fitted parameters tuple from dist.fit()
    title  : plot title
    xlabel : x axis label
    """
    x = np.linspace(data.min(), data.max(), 1000)
    ecdf = stats.ecdf(data)
    theoretical_cdf = dist.cdf(x, *params)

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    # Left: histogram with fitted PDF overlaid
    axes[0].hist(data, bins=50, density=True, color='steelblue', alpha=0.6, label='Empirical')
    axes[0].plot(x, dist.pdf(x, *params), color='crimson', linewidth=2, label='Fitted PDF')
    axes[0].set_title(f'{title} — histogram vs fitted PDF')
    axes[0].set_xlabel(xlabel)
    axes[0].set_ylabel('Density')
    axes[0].legend()

    # Right: empirical CDF vs theoretical CDF
    axes[1].plot(x, ecdf.cdf.evaluate(x), color='steelblue', linewidth=2, label='Empirical CDF')
    axes[1].plot(x, theoretical_cdf, color='crimson', linewidth=2, linestyle='--', label='Theoretical CDF')
    axes[1].set_title(f'{title} — empirical vs theoretical CDF')
    axes[1].set_xlabel(xlabel)
    axes[1].set_ylabel('P(X <= x)')
    axes[1].set_ylim(0, 1.05)
    axes[1].legend()

    plt.tight_layout()
    plt.savefig(f'{title.lower().replace(" ", "_")}_fit.png', dpi=150, bbox_inches='tight')
    plt.show()


def goodness_of_fit(data: np.ndarray, dist, params: tuple, dist_name: str):
    """
    Run Kolmogorov-Smirnov goodness of fit test.
    Tests whether data is consistent with the fitted theoretical distribution.
    
    H0: data follows the specified distribution
    H1: data does not follow the specified distribution
    
    p > 0.05 -> cannot reject H0 (data is consistent with distribution)
    p < 0.05 -> reject H0 (data likely does not follow this distribution)
    """
    ks_stat, p_value = stats.kstest(data, dist.cdf, args=params)
    print(f'--- Kolmogorov-Smirnov goodness of fit: {dist_name} ---')
    print(f'KS statistic: {ks_stat:.4f}')
    print(f'p-value:      {p_value:.4f}')
    if p_value > 0.05:
        print(f'Conclusion:   Cannot reject that data follows {dist_name} (p > 0.05)')
    else:
        print(f'Conclusion:   Data likely does NOT follow {dist_name} (p <= 0.05)')
    return ks_stat, p_value

---
## 1. Log-normal distribution
**Real data:** California Housing median income (`MedInc`)

### When to suspect log-normal
- Data is strictly positive and right-skewed
- Values grow multiplicatively rather than additively
- Common in: income, house prices, revenue, session duration, file sizes

### Why income is log-normal
Income differences compound — a 10% raise on a high salary produces a larger absolute gain than 10% on a low salary. Multiplicative processes naturally produce log-normal distributions.

### The key diagnostic
If $X$ is log-normal, then $\log(X)$ is normal. This is both the definition and the fastest visual check.

### Formula
$$f(x; \mu, \sigma) = \frac{1}{x \sigma \sqrt{2\pi}} \exp\left(-\frac{(\ln x - \mu)^2}{2\sigma^2}\right), \quad x > 0$$

Where $\mu$ and $\sigma$ are the mean and standard deviation of $\ln(X)$, not of $X$ itself.

In [ ]:
income = housing['MedInc'].values

# Step 1 — visual diagnostic: does log(income) look normal?
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(income, bins=50, density=True, color='steelblue', alpha=0.8)
axes[0].set_title('MedInc — raw (right-skewed = log-normal candidate)')
axes[0].set_xlabel('Median income ($10,000s)')
axes[0].set_ylabel('Density')

axes[1].hist(np.log(income), bins=50, density=True, color='darkorange', alpha=0.8)
axes[1].set_title('log(MedInc) — if this looks normal, data is log-normal')
axes[1].set_xlabel('log(Median income)')
axes[1].set_ylabel('Density')

plt.tight_layout()
plt.savefig('lognormal_diagnostic.png', dpi=150, bbox_inches='tight')
plt.show()

print('If the right plot looks approximately bell-shaped, log-normal is a reasonable fit.')

In [ ]:
# Step 2 — fit log-normal distribution using scipy.stats.lognorm
# fit() returns (shape, loc, scale) — for log-normal:
# shape = sigma (std of log(X)), loc = 0 (fix at 0 for strict log-normal), scale = exp(mu)
params_lognorm = stats.lognorm.fit(income, floc=0)  # floc=0 fixes location at 0
shape, loc, scale = params_lognorm

# Recover interpretable parameters
mu_log    = np.log(scale)   # mean of log(X)
sigma_log = shape           # std of log(X)

print('--- Fitted log-normal parameters ---')
print(f'mu (mean of log income):    {mu_log:.4f}')
print(f'sigma (std of log income):  {sigma_log:.4f}')
print(f'\nVerification — sample mean of log(income):  {np.log(income).mean():.4f}')
print(f'Verification — sample std of log(income):   {np.log(income).std():.4f}')

# Step 3 — plot fit
plot_fit(income, stats.lognorm, params_lognorm, 'Log-normal income', 'Median income ($10,000s)')

# Step 4 — goodness of fit test
goodness_of_fit(income, stats.lognorm, params_lognorm, 'log-normal')

In [ ]:
# Q-Q plot — the most powerful visual diagnostic for distribution fit
# If points fall on the diagonal line, the data matches the distribution
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Q-Q plot of raw income vs log-normal
(osm, osr), (slope, intercept, r) = stats.probplot(income, dist=stats.lognorm, sparams=params_lognorm, fit=True)
axes[0].scatter(osm, osr, color='steelblue', alpha=0.3, s=5)
axes[0].plot(osm, slope * np.array(osm) + intercept, color='crimson', linewidth=1.5)
axes[0].set_title(f'Q-Q plot — income vs log-normal (R²={r**2:.4f})')
axes[0].set_xlabel('Theoretical quantiles')
axes[0].set_ylabel('Sample quantiles')

# Q-Q plot of log(income) vs normal — equivalent check
stats.probplot(np.log(income), dist='norm', plot=axes[1])
axes[1].set_title('Q-Q plot — log(income) vs normal')
axes[1].get_lines()[0].set(color='steelblue', alpha=0.3, markersize=3)
axes[1].get_lines()[1].set(color='crimson')

plt.tight_layout()
plt.savefig('lognormal_qq.png', dpi=150, bbox_inches='tight')
plt.show()

print('Points close to the diagonal = good fit.')
print('Deviations in the tails = the distribution underestimates extreme values.')

---
## 2. Exponential distribution

### When to suspect exponential
- Data represents **time between independent events**
- Events occur at a constant average rate
- Common in: inter-arrival times, time between transactions, time between failures, session gaps

### The memoryless property
The exponential distribution has a unique property:

$$P(X > s + t \mid X > s) = P(X > t)$$

The probability of waiting another $t$ minutes is the same regardless of how long you have already waited. The distribution has no memory of the past.

### Formula
$$f(x; \lambda) = \lambda e^{-\lambda x}, \quad x \geq 0$$

Where $\lambda$ is the rate parameter (events per unit time) and the mean is $\frac{1}{\lambda}$.

### Data note
The exponential distribution describes **inter-arrival times** between events from a Poisson process. Real-world examples require event log data with timestamps — which is not available in standard sklearn/seaborn datasets. We simulate a Poisson process here because the relationship between Poisson arrivals and exponential inter-arrival times is the concept itself — and simulation is the most direct way to demonstrate it. The distribution fitting and goodness of fit workflow is identical to what you would apply to real event log data.

In [ ]:
# Simulate a Poisson process: events arriving at rate lambda = 5 per hour
# Inter-arrival times follow Exponential(lambda=5)
np.random.seed(42)
true_rate = 5  # events per hour
n_events  = 2000

# Inter-arrival times are exponentially distributed
inter_arrival_times = np.random.exponential(scale=1/true_rate, size=n_events)

print(f'True rate (lambda):              {true_rate} events/hour')
print(f'True mean inter-arrival time:    {1/true_rate:.4f} hours ({60/true_rate:.1f} minutes)')
print(f'Sample mean inter-arrival time:  {inter_arrival_times.mean():.4f} hours')
print(f'Sample std:                      {inter_arrival_times.std():.4f} hours')
print(f'\nKey property: for exponential, mean == std')
print(f'mean/std ratio: {inter_arrival_times.mean()/inter_arrival_times.std():.4f} (should be close to 1.0)')

In [ ]:
# Fit exponential distribution
# scipy.stats.expon.fit() returns (loc, scale) where scale = 1/lambda
params_expon = stats.expon.fit(inter_arrival_times, floc=0)  # floc=0: time starts at 0
loc_e, scale_e = params_expon
fitted_rate = 1 / scale_e

print(f'--- Fitted exponential parameters ---')
print(f'Fitted rate (lambda):     {fitted_rate:.4f} events/hour')
print(f'Fitted mean (1/lambda):   {scale_e:.4f} hours')
print(f'True rate:                {true_rate} events/hour')

# Plot fit
plot_fit(inter_arrival_times, stats.expon, params_expon, 'Exponential inter-arrival', 'Inter-arrival time (hours)')

# Goodness of fit
goodness_of_fit(inter_arrival_times, stats.expon, params_expon, 'exponential')

In [ ]:
# Demonstrate the memoryless property
# P(X > s + t | X > s) should equal P(X > t)
s, t = 0.1, 0.2  # hours

# Empirical calculation from the data
p_greater_t           = (inter_arrival_times > t).mean()
p_greater_s_plus_t    = (inter_arrival_times > s + t).mean()
p_greater_s           = (inter_arrival_times > s).mean()
p_conditional         = p_greater_s_plus_t / p_greater_s  # conditional probability

print('--- Memoryless property demonstration ---')
print(f'P(X > {t}):                        {p_greater_t:.4f}')
print(f'P(X > {s+t} | X > {s}):           {p_conditional:.4f}')
print(f'\nThese should be approximately equal — and they are.')
print(f'The past waiting time of {s} hours tells you nothing about future waiting time.')

---
## 3. Normal distribution
**Real data:** California Housing `HouseAge`

### When to suspect normal
- Data is symmetric and bell-shaped
- Results from the sum or average of many independent factors
- Common in: measurement errors, aggregated metrics, residuals from regression models

### The Central Limit Theorem (CLT)
The most important theorem in statistics:

$$\bar{X}_n \xrightarrow{d} \mathcal{N}\left(\mu, \frac{\sigma^2}{n}\right) \text{ as } n \to \infty$$

The sample mean of **any** distribution approaches a normal distribution as sample size grows — regardless of what the original distribution looks like. This is why the normal distribution appears everywhere in inference and hypothesis testing.

### Formula
$$f(x; \mu, \sigma) = \frac{1}{\sigma\sqrt{2\pi}} \exp\left(-\frac{(x-\mu)^2}{2\sigma^2}\right)$$

### The 68-95-99.7 rule
$$P(\mu - \sigma \leq X \leq \mu + \sigma) \approx 0.6827$$
$$P(\mu - 2\sigma \leq X \leq \mu + 2\sigma) \approx 0.9545$$
$$P(\mu - 3\sigma \leq X \leq \mu + 3\sigma) \approx 0.9973$$

In [ ]:
house_age = housing['HouseAge'].values

# Fit normal distribution
# norm.fit() returns (mu, sigma) directly
params_norm = stats.norm.fit(house_age)
mu_n, sigma_n = params_norm

print('--- Fitted normal parameters ---')
print(f'mu (mean):    {mu_n:.4f} years')
print(f'sigma (std):  {sigma_n:.4f} years')
print(f'\n--- 68-95-99.7 rule applied to house age ---')
print(f'68% of houses are between {mu_n-sigma_n:.1f} and {mu_n+sigma_n:.1f} years old')
print(f'95% of houses are between {mu_n-2*sigma_n:.1f} and {mu_n+2*sigma_n:.1f} years old')

# Plot fit
plot_fit(house_age, stats.norm, params_norm, 'Normal house age', 'House age (years)')

# Goodness of fit
goodness_of_fit(house_age, stats.norm, params_norm, 'normal')

In [ ]:
# Demonstrate the Central Limit Theorem on real data
# HouseAge is NOT normally distributed — but its sample means WILL be
np.random.seed(42)
sample_sizes = [5, 30, 100]
n_simulations = 5000

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, n in zip(axes, sample_sizes):
    # Draw n_simulations samples of size n and compute each sample mean
    sample_means = [
        np.random.choice(house_age, size=n, replace=True).mean()
        for _ in range(n_simulations)
    ]
    sample_means = np.array(sample_means)

    # Plot histogram of sample means
    ax.hist(sample_means, bins=50, density=True, color='steelblue', alpha=0.7)

    # Overlay fitted normal
    x = np.linspace(sample_means.min(), sample_means.max(), 300)
    mu_clt    = house_age.mean()
    sigma_clt = house_age.std() / np.sqrt(n)
    ax.plot(x, stats.norm.pdf(x, mu_clt, sigma_clt), color='crimson', linewidth=2)
    ax.set_title(f'Sample means (n={n})')
    ax.set_xlabel('Sample mean house age')
    ax.set_ylabel('Density')

plt.suptitle('Central Limit Theorem — sample means converge to normal as n grows', y=1.02, fontsize=12)
plt.tight_layout()
plt.savefig('clt_demonstration.png', dpi=150, bbox_inches='tight')
plt.show()

print('The original HouseAge distribution is NOT normal.')
print('But sample means of any size n become increasingly normal as n grows.')
print('This is why so many statistical tests assume normality — the CLT justifies it.')

---
## 4. Choosing the right distribution — decision framework

| Question | Answer | Candidate distribution |
|---|---|---|
| Is data strictly positive and right-skewed? | Yes | Log-normal, Exponential, Weibull |
| Does log(data) look normal? | Yes | Log-normal |
| Is data time between independent events? | Yes | Exponential |
| Is data symmetric and bell-shaped? | Yes | Normal |
| Is data an average or sum of many things? | Yes | Normal (CLT) |
| Is the hazard rate increasing over time? | Yes | Weibull |
| Is data bounded between 0 and 1? | Yes | Beta |

**Always validate with:**
1. Visual — histogram + fitted PDF, empirical vs theoretical CDF
2. Q-Q plot — most sensitive diagnostic
3. Formal test — Kolmogorov-Smirnov, Shapiro-Wilk (normality only), Anderson-Darling

In [ ]:
# Production utility — fit multiple distributions and compare
def compare_distributions(data: np.ndarray, distributions: list, data_label: str):
    """
    Fit multiple distributions to data and rank by KS statistic.
    Lower KS statistic = better fit.
    
    Parameters
    ----------
    data          : raw observations
    distributions : list of (name, scipy dist, fit_kwargs) tuples
    data_label    : label for display
    """
    results = []
    for name, dist, fit_kwargs in distributions:
        params = dist.fit(data, **fit_kwargs)
        ks_stat, p_value = stats.kstest(data, dist.cdf, args=params)
        results.append({
            'distribution': name,
            'ks_statistic': round(ks_stat, 4),
            'p_value':      round(p_value, 4),
            'params':       params
        })

    results_df = pd.DataFrame(results).sort_values('ks_statistic')
    print(f'--- Distribution comparison for {data_label} ---')
    print(results_df[['distribution', 'ks_statistic', 'p_value']].to_string(index=False))
    print('\nLowest KS statistic = best fit')
    return results_df

# Compare distributions on median income
distributions_to_try = [
    ('log-normal', stats.lognorm, {'floc': 0}),
    ('normal',     stats.norm,    {}),
    ('exponential',stats.expon,   {'floc': 0}),
    ('gamma',      stats.gamma,   {'floc': 0}),
]

results = compare_distributions(income, distributions_to_try, 'median income')

---
## Summary

| Distribution | Shape | Typical use case | scipy tool |
|---|---|---|---|
| Log-normal | Right-skewed, positive | Income, prices, revenue | `stats.lognorm` |
| Exponential | Right-skewed, positive, memoryless | Inter-arrival times | `stats.expon` |
| Normal | Symmetric, bell-shaped | Aggregated metrics, residuals | `stats.norm` |

**The modeling workflow in production:**
1. Plot histogram and empirical CDF
2. Hypothesize candidate distributions from domain knowledge
3. Fit with `dist.fit(data)`
4. Compare empirical CDF vs theoretical CDF visually
5. Run Q-Q plot
6. Confirm with KS test — `stats.kstest(data, dist.cdf, args=params)`
7. If multiple candidates, compare KS statistics with `compare_distributions()`

**Production libraries used:**
- `scipy.stats` — distribution fitting, CDF, PDF, goodness of fit tests
- `numpy` — numerical operations
- `pandas` — results tables
- `matplotlib` / `seaborn` — visualization

**Next:** Chapter 6 — Probability Density Functions (PDF)